<a href="https://colab.research.google.com/github/yilinw762/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [2]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [4]:
# TODO
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print(f'total revenue: ${total_revenue:,.2f}')
print(f'total units: {total_units:,}')

# the total revenue is $8,520.00
# the total units is 783

total revenue: $8,520.00
total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [5]:
# TODO
by_category = (
    df.groupby('category')[['revenue']]
    .sum()
    .sort_values('revenue', ascending=False)
)

by_category['share_pct'] = (
    by_category['revenue'] / total_revenue * 100
)

display(by_category.round(2))

top_category = by_category.index[0]
print(
    f"{top_category} generated the most revenue: "
    f"${by_category.loc[top_category, 'revenue']:,.2f}, "
    f"or {by_category.loc[top_category, 'share_pct']:.1f}% of the total."
)

# Food generated the most revenue: $4,293.00, or 50.4% of the total.
# As we can see down below, rain gear has the least revenue of 10.58% of the total.

,revenue,share_pct
category,,
Food,4293.0,50.39
Merch,1771.5,20.79
Drink,1554.0,18.24
RainGear,901.5,10.58


Food generated the most revenue: $4,293.00, or 50.4% of the total.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [6]:
# TODO
by_vendor = (
    df.groupby('vendor_id')
    .agg(
        avg_order_revenue=('revenue', 'mean'),
        order_count=('revenue', 'size')
    )
    .sort_values('avg_order_revenue', ascending=False)
)

display(by_vendor.round(2))

top_vendor = by_vendor.index[0]
print(
    f"{top_vendor} had the highest average order revenue at "
    f"${by_vendor.loc[top_vendor, 'avg_order_revenue']:.2f}, "
    f"based on {by_vendor.loc[top_vendor, 'order_count']} orders."
)

# V-01 had the highest average order revenue at $22.60, based on 94 orders.

,avg_order_revenue,order_count
vendor_id,,
V-01,22.60,94
V-18,21.75,108
V-05,20.58,93
V-10,20.31,105


V-01 had the highest average order revenue at $22.60, based on 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [8]:
# TODO
merch_revenue = df.loc[
    df['category'] == 'Merch', 'revenue'
].sum()

merch_share = merch_revenue / total_revenue * 100

print(f"Merch contributed {merch_share:.1f}% of total revenue.")

# Merch contributed 20.8% of total revenue.

Merch contributed 20.8% of total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [9]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

assert len(joined) == len(df), "Row count changed."
assert np.isclose(
    joined['revenue'].sum(), total_revenue
), "Revenue total changed."

unmatched_ids = joined.loc[
    joined['vendor_name'].isna(), 'vendor_id'
].unique()

print(f"Unmatched vendor IDs: {unmatched_ids.tolist()}")

# Keep the orders and label missing names using their vendor IDs.
joined['vendor_name'] = joined['vendor_name'].fillna(
    'Unknown vendor (' + joined['vendor_id'] + ')'
)

display(joined.head())

print(
    f"The join preserved all {len(joined)} orders and "
    f"${joined['revenue'].sum():,.2f} in revenue."
)

# V-18 was missing from the lookup, so I kept its orders and labeled it Unknown vendor (V-18). Dropping it would remove valid revenue.

Unmatched vendor IDs: ['V-18']


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown vendor (V-18)
2,V-18,Drink,3,4.5,13.5,Unknown vendor (V-18)
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown vendor (V-18)


The join preserved all 400 orders and $8,520.00 in revenue.


**The unmatched vendor, and what I did about it:** _..._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [10]:
# TODO
vendor_report = joined.pivot_table(
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

display(vendor_report.round(2))

print(
    f"The report breaks down revenue by vendor and category, "
    f"with a grand total of "
    f"${vendor_report.loc['Total', 'Total']:,.2f}."
)

# margins=True adds the row and column totals.

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor (V-18),582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


The report breaks down revenue by vendor and category, with a grand total of $8,520.00.


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [11]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a: I would focus on stocking food because it earned $4,293, about half of total revenue. Merch earned $1,771.50, so I would keep enough of that available too. No need to over stock rain gear if weather is not rainy because rain gear did not generate as much revenue.

b: Q6 is less trustworthy because V-18’s vendor name is missing. Its $2,349 is included, but we do not know which business it belongs to.